In [1]:
import faiss
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from ast import literal_eval
from tqdm import tqdm
import requests
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
from scipy.stats import zscore, entropy
import json, os
import Levenshtein
from IPython.display import display, HTML
import difflib, html
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

In [2]:
def highlight_diff(a,b,c1='background:salmon',c2='background:lightgreen'):
  s=difflib.SequenceMatcher(None,a,b)
  h1=[]; h2=[]
  for tag,i1,i2,j1,j2 in s.get_opcodes():
    if tag=='equal':
      h1.append(html.escape(a[i1:i2])); h2.append(html.escape(b[j1:j2]))
    elif tag=='replace':
      h1.append(f"<span style='{c1}'>"+html.escape(a[i1:i2])+"</span>")
      h2.append(f"<span style='{c2}'>"+html.escape(b[j1:j2])+"</span>")
    elif tag=='delete':
      h1.append(f"<span style='{c1}'>"+html.escape(a[i1:i2])+"</span>")
    elif tag=='insert':
      h2.append(f"<span style='{c2}'>"+html.escape(b[j1:j2])+"</span>")
  out=f"<div style='font-family:monospace;white-space:pre-wrap'>"+''.join(h1)+"</div><div style='font-family:monospace;white-space:pre-wrap;margin-top:4px'>"+''.join(h2)+"</div>"
  display(HTML(out))

def get_debate_context(hitloc, tk):
    return tk[pd.to_datetime(tk.date) == hitloc['tk_date']]

def plot_scatter_grid(
    df,
    p_id_highlight=None,
    highlight_word=None,
    highlight_color="yellow",
    n_cols=15
):

    df = df.reset_index(drop=True)

    N = len(df)
    n_rows = int(np.ceil(N / n_cols))

    # --- Colors (party) ---
    parties = df["party-ref"].fillna("UNKNOWN").unique()
    cmap = plt.get_cmap("tab10")
    party_colors = {p: cmap(i % 10) for i, p in enumerate(parties)}

    # --- Markers (role) ---
    marker_list = ["o", "s", "^", "D", "P", "X", "*", "v", "<", ">"]
    roles = df["role"].fillna("UNKNOWN").unique()
    role_markers = {r: marker_list[i % len(marker_list)] for i, r in enumerate(roles)}

    # --- Grid positions ---
    xs = np.arange(N) % n_cols
    ys = np.arange(N) // n_cols
    ys = -ys  # invert so 0 is top row

    fig, ax = plt.subplots(figsize=(6,12))

    # Normalize highlight word for comparison
    word = highlight_word.lower() if highlight_word else None

    # --- Plot scatters ---
    for i, row in df.iterrows():
        x, y = xs[i], ys[i]
        base_color = party_colors[row["party-ref"]]
        marker = role_markers[row["role"]]

        # --- Word highlight ---
        contains_word = False
        if word:
            txt = str(row["text"]).lower()
            contains_word = word.lower() in txt

        # --- Category: word highlight takes precedence over party color ---
        if contains_word:
            color = base_color
            size = 150
            edge = "black"
            width = 1.2,
            ls = 'dashed'

        # --- p_id highlight (if not already highlighted by word) ---
        elif row["p_id"] == p_id_highlight:
            color = base_color
            size = 300
            edge = "black"
            width = 1.5
            ls='solid'

        else:
            color = base_color
            size = 50
            edge = "none"
            width = 0.5
            ls='solid'

        ax.scatter(
            x, y,
            s=size,
            c=[color],
            marker=marker,
            edgecolor=edge,
            linewidth=width,
            linestyle=ls
        )

    # --- Aesthetics ---
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-(n_rows), 0.5)
    ax.set_xticks(range(n_cols))
    ax.set_yticks([])

    ax.set_title(
        "Speech Grid — Color = Party, Marker = Role\n"
        "(Highlighted p_id and word matches)"
    )

    # --- Legends ---
    # Party legend
    party_handles = [
        plt.Line2D([0], [0], marker='o', color='white',
                   markerfacecolor=party_colors[p], markersize=10,
                   linestyle='') for p in parties
    ]
    party_legend = ax.legend(party_handles, parties, title="Party", loc="upper right")
    ax.add_artist(party_legend)

    # Role legend
    role_handles = [
        plt.Line2D([0], [0], marker=role_markers[r], color='black',
                   linestyle='', markersize=10) for r in roles
    ]
    ax.legend(role_handles, roles, title="Role", loc="lower right")

    plt.tight_layout()
    plt.show()

In [3]:
tk = pd.read_csv("data/tk.csv",sep='\t')

/tmp/ipykernel_286406/3443076796.py:1: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  tk = pd.read_csv("data/tk.csv",sep='\t')


In [4]:
with open('data/meta-reports.json', 'r') as f:
    report_ids = json.load(f)
    report_mtd = {rid['_id']:rid for rid in report_ids}

In [5]:
def word_in_date(word, tk):
    results = {}

    for date, df in tk.groupby(pd.to_datetime(tk.date)):
        tx = " ".join(df.text).lower()
        results[date] = True if word in tx else False
    return results

def get_activiteit_by_date(date: pd.Timestamp) -> dict:
    """
    Fetch the first plenair debat activity for a given date from the Tweede Kamer OData API.

    Args:
        date (pd.Timestamp): The date to filter on.

    Returns:
        dict: The first activity result as a dictionary, or None if no results.
    """
    # Build URL with date filter
    base_url = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Activiteit"
    date_filter = (
        f"year(Datum) eq {date.year} "
        f"and month(Datum) eq {date.month} "
        f"and day(Datum) eq {date.day}"
    )

    odata_filter = f"contains(soort,'plenair debat') and Status eq 'Uitgevoerd' and {date_filter}"

    # Encode URL
    url = f"{base_url}?$filter={requests.utils.quote(odata_filter)}&$count=true&expand=Document"

    # Request API
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    # Return first result if exists
    if data.get("value"):
        return data["value"][0]
    return None

org_in_date = word_in_date("rekenkamer", tk)

In [20]:
dfm = pd.read_csv('results/matches-rep-kst-chunks.csv')
dfm['rep_sentence_indices'] = dfm['rep_sentence_indices'].apply(literal_eval)
dfm['kst_sentence_indices'] = dfm['tk_sentence_indices'].apply(literal_eval)

In [21]:
dfm = dfm.sort_values("score", ascending=False)

# Keep track of sentence indices we've already included
seen_sentences = set()
rows_to_keep = []

for idx, row in dfm.iterrows():
    # Check if any sentence index in this row has been seen
    if not any(s in seen_sentences for s in row["rep_sentence_indices"]):
        rows_to_keep.append(idx)
        seen_sentences.update(row["rep_sentence_indices"])

# Filter the dataframe
dfm = dfm.loc[rows_to_keep].reset_index(drop=True)

In [27]:
dfm.loc[10344]['rep_chunk_text'],dfm.loc[10344]['tk_chunk_text']

('Wat het automatisch continueren betreft (dat wil zeggen dat huursubsidieontvangers niet meer jaarlijks een aanvraag [ORG] 10 hoeven in te dienen) is de doelmatigheidswinst van de lokale partners vastgelegd en in de meerjarenbegroting verwerkt. Voor het proces eerste aanvraag (een reactie op een eerste aanvraag zoveel mogelijk binnen vier weken) dat op zijn vroegst per 1 januari 2002 van start gaat, is deze exercitie vanwege de nog bestaande onzekerheden nog niet gemaakt. [ORG] 11 7 Huursubsidie als inkomens instrument Aparte aandacht verdient naar het oordeel van de [ORG] de relatie van het huursubsidiebeleid met het algemene inkomensbeleid. De [ORG] heeft in haar onderzoek geconstateerd dat de jaarlijkse aanpassingen van het huursubsidiebeleid in de afgelopen jaren regelmatig verbonden waren met over wegingen van inkomenspolitieke aard.',
 'Daarnaast is de verhoging in het tweede halfjaar van 2007 gecompenseerd (Besluit van 21 januari 2008, Stb. 1.2.3 Formule huurtoeslagberekening B

In [ ]:
dfm['lvs'] = [Levenshtein.ratio(s,t) for s,t in tqdm(zip(dfm.rep_sentence, dfm.tk_sentence))]
dfm['dydiff'] = (dfm.tk_date - dfm.rep_date ).dt.days
# dfm['court_in_debate'] = dfm.tk_date.map(org_in_date)
dfm['sem_syn_diff'] = dfm.score - dfm.lvs

In [10]:
dfms = dfm[
        (dfm.score > .85) &
        (dfm.dydiff > 0) &
        (dfm.dydiff < 365) &
        (dfm.lvs > .5) &
        (dfm.tk_sentence != dfm.rep_sentence)
        # (dfm.court_in_debate == False) &
        # ~(dfm.tk_sentence.str.contains('ORG'))
         ]

print(dfms.shape)

(218, 20)


In [ ]:
dfms[['rep_sentence',]]

,score,rep_id,rep_date,rep_detail_url,rep_sentence,rep_sent_id,rep_sid,tk_id,tk_source,tk_date,tk_detail_url,tk_sub_source,tk_source_title,tk_source_subject,tk_sentence,tk_sent_id,tk_sid,lvs,dydiff,sem_syn_diff
4318,0.851849,rekenkamer__kamerstuk:2012:05:16:antwoorden-op...,2012-05-16,https://www.rekenkamer.nl/publicaties/kamerstu...,Wij verwijzen u naar ons antwoord op vraag 4 e...,77,rekenkamer__kamerstuk:2012:05:16:antwoorden-op...,kamerstukken__schriftelijke_vragen_antwoord:17...,kamerstukken__schriftelijke_vragen_antwoord,2013-03-29,https://www.tweedekamer.nl/kamerstukken/detail...,Antwoord schriftelijke vragen,NaN,Antwoord vragen van de leden Marcouch en Heijn...,Daarvoor verwijzen wij u naar het antwoord op ...,6,kamerstukken__schriftelijke_vragen_antwoord:17...,0.689076,317,0.162773
30298,0.868827,rekenkamer__kamerstuk:2013:02:01:beantwoording...,2013-02-01,https://www.rekenkamer.nl/publicaties/kamerstu...,Het schuifplan Ermelo betreft een complex proj...,62,rekenkamer__kamerstuk:2013:02:01:beantwoording...,kamerstukken__algemeen_overleg:4416481f-2d89-4...,kamerstukken__algemeen_overleg,2013-03-14,https://www.tweedekamer.nl/kamerstukken/detail...,Verslag van een algemeen overleg,Beleidsbrief Defensie,"Verslag van een algemeen overleg, gehouden op ...",Het betreft een complex project met aanzienlij...,85,kamerstukken__algemeen_overleg:4416481f-2d89-4...,0.917431,41,-0.048604
53629,0.964763,rekenkamer__kamerstuk:2013:11:28:europees-econ...,2013-11-28,https://www.rekenkamer.nl/publicaties/kamerstu...,Indien de nationale controle-instanties in sam...,129,rekenkamer__kamerstuk:2013:11:28:europees-econ...,kamerstukken__brief_regering:f7cc817c-7b3b-423...,kamerstukken__brief_regering,2014-02-13,https://www.tweedekamer.nl/kamerstukken/detail...,Brief regering,Verbetering verantwoording en begroting,Aandachtspunten van Algemene Rekenkamer over E...,Indien de nationale rekenkamers in samenwerkin...,53,kamerstukken__brief_regering:f7cc817c-7b3b-423...,0.965649,77,-0.000886
54729,0.964763,rekenkamer__kamerstuk:2013:11:28:europees-econ...,2013-11-28,https://www.rekenkamer.nl/publicaties/kamerstu...,Indien de nationale controle-instanties in sam...,129,rekenkamer__kamerstuk:2013:11:28:europees-econ...,kamerstukken__brief_regering:f7cc817c-7b3b-423...,kamerstukken__brief_regering,2014-02-13,https://www.tweedekamer.nl/kamerstukken/detail...,Brief regering,Verbetering verantwoording en begroting,Aandachtspunten van Algemene Rekenkamer over E...,Indien de nationale rekenkamers in samenwerkin...,53,kamerstukken__brief_regering:f7cc817c-7b3b-423...,0.965649,77,-0.000886
63487,0.973853,rekenkamer__kamerstuk:2014:04:02:beantwoording...,2014-04-02,https://www.rekenkamer.nl/publicaties/kamerstu...,intern personeel (handleiding overheidstarieve...,5,rekenkamer__kamerstuk:2014:04:02:beantwoording...,kamerstukken__brief_regering:ef24e0ee-bc4e-4da...,kamerstukken__brief_regering,2014-10-16,https://www.tweedekamer.nl/kamerstukken/detail...,Brief regering,Project SPEER,Voortgangsrapportage basisimplementatie ERP ov...,Intern personeel (handleiding overheidstarieve...,50,kamerstukken__brief_regering:ef24e0ee-bc4e-4da...,0.986425,197,-0.012572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1178342,0.877166,rekenkamer__rapport:2024:02:06:focus-op-corona...,2024-02-06,https://www.rekenkamer.nl/publicaties/rapporte...,Ondernemers zijn niet per brief geïnformeerd o...,148,rekenkamer__rapport:2024:02:06:focus-op-corona...,kamerstukken__brief_regering:fbf0022a-8d21-4ad...,kamerstukken__brief_regering,2024-06-25,https://www.tweedekamer.nl/kamerstukken/detail...,Brief regering,Jaarverslag en slotwet Ministerie van Financië...,Beantwoording vragen commissie V100 over coron...,Ondernemers zijn niet individueel per brief ge...,25,kamerstukken__brief_regering:fbf0022a-8d21-4ad...,0.880000,140,-0.002834
1200614,0.899983,rekenkamer__rapport:2024:05:15:resultaten-vera...,2024-05-15,https://www.rekenkamer.nl/publicaties/rapporte.

In [16]:
# Write to Markdown file
with open("/home/rb/Documents/Code/Rekenkamer/results/formatted_matches_rep_tk.md", "w", encoding="utf-8") as f:
    for i, r in dfms.sort_values('score').iterrows():
        try:
            f.write('---\n\n')
            f.write(f'### HIT — SCORE: **{round(r["score"], 2)}\n**')
            f.write(f'### HIT — LVS: **{round(r["lvs"], 2)}**\n')

            f.write(f'**REPORT SENTENCE:** {r['rep_sentence']}\n')
            f.write(f'**TK SENTENCE:** {r['tk_sentence']}\n\n')

            rp_mtd = report_mtd.get(r['rep_id'])

            # Report info
            f.write(f'**Report Title:** {rp_mtd.get("title")}\n')
            f.write(f'**Report Date:** {rp_mtd.get("date")}\n')
            f.write(f'**Report Department:** {rp_mtd.get("category")}\n\n')

            # Debate info
            f.write(f'**Debate Date:** {str(r['tk_date'])}\n\n')

        except Exception as e:
            continue


In [199]:
dfms

,score,rep_id,rep_date,rep_detail_url,rep_sentence,rep_sent_id,rep_sid,tk_id,tk_member-ref,tk_party-ref,tk_role,tk_date,tk_house,tk_sentence,tk_sent_id,tk_sid,lvs,dydiff,court_in_debate,sem_syn_diff
788,0.663900,rekenkamer__kamerstuk:2012:05:03:beantwoording...,2012-05-03,https://www.rekenkamer.nl/publicaties/kamerstu...,Het is niet aan de [ORG] over die wenselijkhei...,114,rekenkamer__kamerstuk:2012:05:03:beantwoording...,ParlaMint-NL_2020-02-04-tweedekamer-36.u172,NaN,VVD,government,2020-02-04,commons,Het is niet aan de overheid om zich over die k...,2,ParlaMint-NL_2020-02-04-tweedekamer-36.u172_s2,0.709220,2833,False,-0.045320
2263,0.723975,rekenkamer__kamerstuk:2012:09:20:controle-van-...,2012-09-20,https://www.rekenkamer.nl/publicaties/kamerstu...,Een afschrift van deze brief sturen wij aan de...,43,rekenkamer__kamerstuk:2012:09:20:controle-van-...,ParlaMint-NL_2017-07-04-tweedekamer-26.u73,NaN,PvdA,mp,2017-07-04,commons,Wij hebben hierover een brief gehad van de min...,2,ParlaMint-NL_2017-07-04-tweedekamer-26.u73_s2,0.607477,1748,False,0.116498
4051,0.666602,rekenkamer__kamerstuk:2012:12:18:vastgoedplan-...,2012-12-18,https://www.rekenkamer.nl/publicaties/kamerstu...,"Een motie die de minister daartoe oproept, wer...",73,rekenkamer__kamerstuk:2012:12:18:vastgoedplan-...,ParlaMint-NL_2017-12-21-tweedekamer-10.u19,NaN,VVD,unknown,2017-12-21,commons,Die motie heb ik destijds ontraden en die moti...,2,ParlaMint-NL_2017-12-21-tweedekamer-10.u19_s2,0.663158,1829,False,0.003444
4399,0.660445,rekenkamer__kamerstuk:2013:02:14:beantwoording...,2013-02-14,https://www.rekenkamer.nl/publicaties/kamerstu...,Antwoord De [ORG] is niet in de positie om dez...,58,rekenkamer__kamerstuk:2013:02:14:beantwoording...,ParlaMint-NL_2015-03-26-tweedekamer-26.u43,NaN,PvdA,unknown,2015-03-26,commons,Het is niet aan mij om op deze vraag een antwo...,2,ParlaMint-NL_2015-03-26-tweedekamer-26.u43_s2,0.635659,770,False,0.024786
4636,0.694061,rekenkamer__kamerstuk:2013:03:01:beantwoording...,2013-03-01,https://www.rekenkamer.nl/publicaties/kamerstu...,Uit verschillende recente studies blijkt dat d...,16,rekenkamer__kamerstuk:2013:03:01:beantwoording...,ParlaMint-NL_2016-01-27-tweedekamer-7.u42,NaN,GL,mp,2016-01-27,commons,Die rapporten geven nu juist aan dat met het h...,1,ParlaMint-NL_2016-01-27-tweedekamer-7.u42_s1,0.563107,1062,False,0.130954
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259384,0.695324,rekenkamer__rapport:2020:05:20:staat-van-de-ri...,2020-05-20,https://www.rekenkamer.nl/publicaties/rapporte...,En soms wordt er beweerd dat dit de grootste c...,13,rekenkamer__rapport:2020:05:20:staat-van-de-ri...,ParlaMint-NL_2020-05-28-tweedekamer-3.u163,NaN,VVD,unknown,2020-05-28,commons,We zitten op dit moment namelijk in misschien ...,3,ParlaMint-NL_2020-05-28-tweedekamer-3.u163_s3,0.681319,8,False,0.014005
260365,0.838307,rekenkamer__rapport:2020:05:20:staat-van-de-ri...,2020-05-20,https://www.rekenkamer.nl/publicaties/rapporte...,"Denk aan de personeelstekorten in de zorg, in ...",75,rekenkamer__rapport:2020:05:20:staat-van-de-ri...,ParlaMint-NL_2020-09-01-tweedekamer-3.u15,NaN,GL,unknown,2020-09-01,commons,"Denk aan de zorg, denk aan het onderwijs, denk...",4,ParlaMint-NL_2020-09-01-tweedekamer-3.u15_s4,0.681159,104,False,0.157148
261162,0.656090,rekenkamer__rapport:2020:06:25:geen-plek-voor-...,2020-06-25,https://www.rekenkamer.nl/publicaties/rapporte...,De [ORG] is van oordeel dat dit een stap in de...,32,rekenkamer__rapport:2020:06:25:geen-plek-voor-...,ParlaMint-NL_2022-01-27-tweedekamer-7.u3,NaN,D66,government,2022-01-27,commons,Ik denk dat dit wetsvoorstel ook een hele goed...,22,ParlaMint-NL_2022-01-27-tweedekamer-7.u3_s22,0.540146,581,False,0.115944
261165,0.669757,rekenkamer__rapport:2020:06:25:geen-plek-voor-...,2020-06-25,https://www.rekenkamer.nl/publicaties/rapporte...,De [ORG] is van oordeel dat dit een stap in de...,32,rekenkamer__rapport:2020:06:25:geen-plek-voor-...,ParlaMint-NL_2020-06-30-

In [197]:
tk.set_index('p_id').loc['nl.proc.ob.d.h-tk-20092010-50-4666.1.7.126.6']['text']

'De heer Van der Vlies en anderen vragen zich af of je met dat oormerken niet in strijd handelt met de decentralisatiegedachte als zodanig die onder de Wmo lag. Ik vind dat dit niet het geval is. Als het Rijk decentrale overheden zoals gemeenten financieel in staat wil stellen om beleid uit te voeren, kan het ook kiezen voor een specifieke of algemene uitkering. Beide keuzes zijn er. In 2009 waren er 109 specifieke uitkeringen. Voor het ontwikkelen van landschappen, beleid voor onderwijsachterstanden, de wetten rondom de bijstand en de participatiebudgetten zijn er geoormerkte gelden in het Gemeentefonds. Zo gek is het dus niet. Bij de zorggelden is te zien dat er voor daklozenopvang ook een specifieke uitkering is en dat geldt ook voor de jeugdzorg. Er wordt dus veelvuldig gewerkt met een specifieke uitkering, ook als er gedecentraliseerd is.'